# Deep Learning CIA (2087101): Backpropagation and Optimizer Comparison

**Objectives**
1. Implement a one-hidden-layer MLP from scratch using NumPy, MSE loss, analytical backpropagation and standard Gradient Descent.
2. Compare SGD with momentum and Adam on a synthetic classification dataset using PyTorch.
3. Record losses, convergence curves and a short interpretation.

Run all cells from top to bottom. Fixed seeds improve reproducibility; exact results may vary across environments.

## Part 1 — NumPy backpropagation from scratch

The network is input → sigmoid hidden layer → sigmoid output layer. MSE is used as the loss.

In [3]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
X = np.array([[0,0],[0,1],[1,0],[1,1],[.2,.8],[.8,.2],[.1,.1],[.9,.9]], dtype=float)
y = np.array([[.05],[.85],[.85],[.95],[.70],[.70],[.10],[.90]], dtype=float)

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def mse(a, b):
    return np.mean((a-b)**2)

rng = np.random.default_rng(42)
W1 = rng.normal(0, .5, (2, 4)); b1 = np.zeros((1, 4))
W2 = rng.normal(0, .5, (4, 1)); b2 = np.zeros((1, 1))
lr, epochs = .8, 5000
losses = []

for epoch in range(epochs):
    # Forward pass
    a1 = sigmoid(X @ W1 + b1)
    yhat = sigmoid(a1 @ W2 + b2)
    losses.append(mse(y, yhat))

    # Backpropagation for mean squared error
    dyhat = (2 / y.size) * (yhat - y)
    dz2 = dyhat * yhat * (1-yhat)
    dW2 = a1.T @ dz2
    db2 = dz2.sum(axis=0, keepdims=True)

    da1 = dz2 @ W2.T
    dz1 = da1 * a1 * (1-a1)
    dW1 = X.T @ dz1
    db1_grad = dz1.sum(axis=0, keepdims=True)

    # Standard Gradient Descent update
    W2 -= lr*dW2; b2 -= lr*db2
    W1 -= lr*dW1; b1 -= lr*db1_grad

print(f"Initial MSE: {losses[0]:.6f}")
print(f"Final MSE:   {losses[-1]:.6f}")
plt.figure(figsize=(7,4))
plt.plot(losses)
plt.xlabel("Epoch"); plt.ylabel("MSE")
plt.title("NumPy MLP: Backpropagation with Gradient Descent")
plt.grid(True); plt.tight_layout(); plt.show()
print("Predictions [target, prediction]:")
final_hidden = sigmoid(X @ W1 + b1)
final_pred = sigmoid(final_hidden @ W2 + b2)
print(np.round(np.column_stack((y, final_pred)), 4))

Initial MSE: 0.107561
Final MSE:   0.002959
Predictions [target, prediction]:
[[0.05   0.0464]
 [0.85   0.7723]
 [0.85   0.7722]
 [0.95   0.9391]
 [0.7    0.7724]
 [0.7    0.7724]
 [0.1    0.1088]
 [0.9    0.9299]]


### Gradient equations

Let \(Z_1=XW_1+b_1\), \(A_1=\sigma(Z_1)\), \(Z_2=A_1W_2+b_2\), and \(\hat Y=\sigma(Z_2)\).

For \(L=\frac{1}{N}\sum_i(\hat y_i-y_i)^2\):
- \(dZ_2=\frac{2}{N}(\hat Y-Y)\odot\hat Y(1-\hat Y)\)
- \(dW_2=A_1^T dZ_2,\quad db_2=\sum dZ_2\)
- \(dZ_1=(dZ_2W_2^T)\odot A_1(1-A_1)\)
- \(dW_1=X^T dZ_1,\quad db_1=\sum dZ_1\)

Update rule: \(W \leftarrow W-\eta\nabla_W L\), where \(\eta\) is the learning rate.

## Part 2 — Compare SGD with momentum and Adam using PyTorch

The same feedforward architecture and synthetic two-moons dataset are used for both optimizers. Only the optimizer differs.

In [6]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 123
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

X_np, y_np = make_moons(n_samples=2000, noise=.20, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X_np, y_np, test_size=.2, random_state=SEED, stratify=y_np
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
Xte = torch.tensor(X_test, dtype=torch.float32)
yte = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)

loader = DataLoader(
    TensorDataset(Xtr, ytr), batch_size=64, shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)

class FeedforwardNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2,32), nn.ReLU(),
            nn.Linear(32,16), nn.ReLU(),
            nn.Linear(16,1)
        )
    def forward(self, x):
        return self.net(x)

def run_experiment(kind, epochs=100):
    torch.manual_seed(SEED)  # same starting weights
    model = FeedforwardNet()
    loss_fn = nn.BCEWithLogitsLoss()
    if kind == "SGD with momentum":
        opt = torch.optim.SGD(model.parameters(), lr=.05, momentum=.9)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=.01)

    train_losses, test_losses = [], []
    first_below = None
    for ep in range(epochs):
        model.train()
        sum_loss, count = 0., 0
        for xb, yb in loader:
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            sum_loss += loss.item()*len(xb); count += len(xb)
        train_losses.append(sum_loss/count)

        model.eval()
        with torch.no_grad():
            test_loss = loss_fn(model(Xte), yte).item()
        test_losses.append(test_loss)
        if first_below is None and test_loss < .1:
            first_below = ep+1

    with torch.no_grad():
        pred = (torch.sigmoid(model(Xte)) >= .5).float()
        accuracy = (pred == yte).float().mean().item()
    return {
        "model":model, "train":train_losses, "test":test_losses,
        "final_train":train_losses[-1], "final_test":test_losses[-1],
        "accuracy":accuracy, "epoch_below_0_1":first_below
    }

sgd = run_experiment("SGD with momentum")
adam = run_experiment("Adam")

for name, result in [("SGD with momentum", sgd), ("Adam", adam)]:
    print(f"\n{name}")
    print(f"  Final training loss: {result['final_train']:.5f}")
    print(f"  Final test loss:     {result['final_test']:.5f}")
    print(f"  Test accuracy:       {result['accuracy']:.4f}")
    print(f"  First test-loss epoch below 0.1: {result['epoch_below_0_1'] or 'not reached'}")

plt.figure(figsize=(8,4.5))
plt.plot(sgd["test"], label="SGD with momentum")
plt.plot(adam["test"], label="Adam")
plt.xlabel("Epoch"); plt.ylabel("Test BCE loss")
plt.title("Optimizer convergence comparison")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


SGD with momentum
  Final training loss: 0.07338
  Final test loss:     0.09040
  Test accuracy:       0.9675
  First test-loss epoch below 0.1: 7

Adam
  Final training loss: 0.07366
  Final test loss:     0.09907
  Test accuracy:       0.9575
  First test-loss epoch below 0.1: 5


## Results and discussion

Compare the printed final training loss, test loss, accuracy and loss curves.

- **Convergence speed:** Compare the epoch when a chosen loss threshold is first reached, if it is reached.
- **SGD with momentum:** Momentum smooths updates by accumulating a moving direction from past gradients.
- **Adam:** Uses adaptive estimates of gradient first and second moments and often converges quickly in early training.
- **Interpretation:** Lower final test loss is better for this run. Outcomes depend on learning rates, initialization, data and training settings; this experiment does not prove one optimizer is always superior.

## Conclusion

The first experiment demonstrates chain-rule gradient computation and standard Gradient Descent updates. The second experiment compares SGD with momentum and Adam under a controlled setup, using loss logs and convergence curves as evidence.